In [13]:
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from model import FireForecast, MemoryAdapter


In [14]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir=Path("data")
learning_rate=1e-4
history_days=None  # None: all observations since the start of this fire.
labels=("active", "burned", "new")

class FireDataset(Dataset):
    def __init__(self, files):
        self.files=files
        self.cache=None

    def __len__(self):
        return len(self.files)-1

    def read_day(self, index):
        if self.cache is not None:
            return self.cache[index]
        with np.load(self.files[index]) as data:
            return {name: data[name] for name in ("image", "date", *labels)}

    def load_cache(self):
        self.cache=[self.read_day(i) for i in range(len(self.files))]

    def clear_cache(self):
        self.cache=None

    def __getitem__(self, index):
        stop=index+1
        start=0 if history_days is None else max(0, stop-history_days)
        days=[self.read_day(i) for i in range(start, stop)]
        history=torch.from_numpy(np.stack([day["image"] for day in days]))
        dates=torch.tensor([day["date"].item() for day in days], dtype=torch.float32)
        target=self.read_day(stop)
        targets={name: torch.tensor(target[name], dtype=torch.float32) for name in labels}
        return history, dates-dates[-1], targets


In [15]:
incidents=sorted(path for path in data_dir.iterdir()
                 if path.is_dir() and len(list(path.glob("*.npz")))>1)
np.random.default_rng(42).shuffle(incidents)
split=max(1, int(0.8*len(incidents)))

train_loaders=[DataLoader(FireDataset(sorted(folder.glob("*.npz"))), batch_size=1, shuffle=False, num_workers=4, prefetch_factor=1, pin_memory=True) for folder in incidents[:split]]
val_loaders=[DataLoader(FireDataset(sorted(folder.glob("*.npz"))), batch_size=1, shuffle=False, num_workers=4, prefetch_factor=1, pin_memory=True) for folder in incidents[split:]]
train_data=train_loaders[0].dataset

In [16]:
memory=MemoryAdapter(high=1.0, low=-1.0, dims=3, size=4096, device=device,shape={"encoder": 512}, steps=8,)
network=FireForecast(input_channels=2*train_data[0][0].shape[1],memory=memory).to(device)
optimizer=torch.optim.AdamW(network.parameters(), lr=learning_rate, weight_decay=1e-4)


In [17]:
epochs=30
patience=7
min_delta=0.001
noimprove=0
best_val_loss=float("inf")
losses=[]

checkpoint_dir=Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoints=[path for path in checkpoint_dir.glob("epoch_*.pt")
             if path.stem.removeprefix("epoch_").isdigit()]
resume_from=max(checkpoints, key=lambda path: int(path.stem.removeprefix("epoch_")), default=None)
start_epoch=0

if resume_from is not None:
    checkpoint=torch.load(resume_from, map_location="cpu", weights_only=True)
    network.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch=checkpoint["epoch"]
    best_val_loss=checkpoint["best_val_loss"]
    noimprove=checkpoint["noimprove"]
    losses=checkpoint["losses"]
    if checkpoint.get("history_days", 5) != history_days:
        best_val_loss=float("inf")
        noimprove=0  # Different evaluation windows; retain weights, optimizer and epoch.
    del checkpoint
    print(f"Resuming {resume_from.name}; next epoch: {start_epoch+1}.")


In [ ]:
memory=network.memory.memory
memory.voltage=torch.zeros_like(memory.voltage)
memory.spikes=torch.zeros_like(memory.spikes)
memory.y={key: memory.voltage.new_zeros(width) for key, width in memory.shape.items()}

for epoch in range(start_epoch, epochs):
    if noimprove>=patience:
        break
    print(f"Epoch {epoch+1}/{epochs}:")
    for phase, loaders in (("train", train_loaders), ("val", val_loaders)):
        training=phase=="train"
        network.train(training)
        total_loss=0.0
        samples=0
        order=np.random.permutation(len(loaders)) if training else range(len(loaders))

        with torch.set_grad_enabled(training):
            for index in order:
                loader=loaders[index]
                try:
                    loader.dataset.load_cache()
                    for history, dates, targets in loader:
                        history, dates=history.to(device, non_blocking=True), dates.to(device, non_blocking=True)
                        valid=torch.isfinite(history)
                        history=torch.cat([torch.where(valid, history, 0), valid.float()], dim=2)
                        del valid
                        targets={name: target.to(device, non_blocking=True) for name, target in targets.items()}

                        memory.voltage=memory.voltage.detach()
                        memory.spikes=memory.spikes.detach()
                        memory.y={key: value.detach() for key, value in memory.y.items()}

                        if training:
                            optimizer.zero_grad()
                        active, burned, newly_burned=network(history, dates)
                        loss=F.binary_cross_entropy(active, targets["active"])+ F.binary_cross_entropy(burned, targets["burned"])+ 0.5*F.binary_cross_entropy(newly_burned, targets["new"])
                        if training:
                            loss.backward()
                            torch.nn.utils.clip_grad_norm_(network.parameters(), 1.0)
                            optimizer.step()
                        total_loss+=loss.item()
                        samples+=1
                finally:
                    loader.dataset.clear_cache()

                memory.voltage=torch.zeros_like(memory.voltage)
                memory.spikes=torch.zeros_like(memory.spikes)
                memory.y={key: memory.voltage.new_zeros(width) for key, width in memory.shape.items()}

        average_loss=total_loss/samples
        print(f"{phase} loss: {average_loss:.4f}")
        if phase=="val":
            val_loss=average_loss

    losses.append(val_loss)
    if val_loss<best_val_loss-min_delta:
        best_val_loss=val_loss
        noimprove=0
        torch.save(network.state_dict(), "best.pt")
    else:
        noimprove+=1
        print(f"Early stopping: {noimprove}/{patience} epochs without improvement.")

    if (epoch+1)%5==0:
        checkpoint_path=checkpoint_dir / f"epoch_{epoch+1:03d}.pt"
        torch.save({
            "epoch": epoch+1,
            "history_days": history_days,
            "model": network.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_val_loss": best_val_loss,
            "noimprove": noimprove,
            "losses": losses,
        }, checkpoint_path.with_suffix(".tmp"))
        checkpoint_path.with_suffix(".tmp").replace(checkpoint_path)
        print(f"Saved {checkpoint_path}")

    if noimprove>=patience:
        print("Early stopped.")
        break

Epoch 1/30:
train loss: 0.6397
val loss: 0.3276
Epoch 2/30:
train loss: 0.2224
val loss: 0.1455
Epoch 3/30:
train loss: 0.1118
val loss: 0.0820
Epoch 4/30:
train loss: 0.0688
val loss: 0.0544
Epoch 5/30:
train loss: 0.0474
val loss: 0.0515
Saved checkpoints/epoch_005.pt
Epoch 6/30:
train loss: 0.0361
val loss: 0.0281
Epoch 7/30:
train loss: 0.0288
val loss: 0.0210
Epoch 8/30:
train loss: 0.0251
val loss: 0.0229
Early stopping: 1/7 epochs without improvement.
Epoch 9/30:
train loss: 0.0194
val loss: 0.0155
Epoch 10/30:
train loss: 0.0211
val loss: 0.0146
Early stopping: 1/7 epochs without improvement.
Saved checkpoints/epoch_010.pt
Epoch 11/30:
train loss: 0.0164
val loss: 0.0133
Epoch 12/30:
train loss: 0.0153
val loss: 0.0107
Epoch 13/30:
train loss: 0.0154
val loss: 0.0126
Early stopping: 1/7 epochs without improvement.
Epoch 14/30:
train loss: 0.0138
val loss: 0.0094
Epoch 15/30:
train loss: 0.0162
val loss: 0.0087
Early stopping: 1/7 epochs without improvement.
Saved checkpoints/ep

In [ ]:
torch.save(network.state_dict(), "latest.pt")
network.load_state_dict(torch.load("best.pt", map_location=device, weights_only=True))